# 🔗 Graph RAG デモ — Neo4j + Ollama（完全ローカル版）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)
[![Python](https://img.shields.io/badge/Python-3.10%2B-blue?logo=python&logoColor=white)](https://www.python.org/)
[![Neo4j](https://img.shields.io/badge/Neo4j-5.18-008CC1?logo=neo4j&logoColor=white)](https://neo4j.com/)
[![Ollama](https://img.shields.io/badge/Ollama-llama3-black)](https://ollama.com/)

## このノートブックで確認できること

> **「1つのユーザークエリがグラフDBへの複数クエリに展開される」** という Graph RAG の動作を、完全ローカル環境で体験します。

| コンポーネント | 技術 | 役割 |
|---|---|---|
| グラフDB | Neo4j Community 5.18 | Cypher クエリの実行 |
| LLM | Ollama + llama3 | クエリ分解・回答生成 |
| 実行環境 | Google Colab | すべてクラウド上で完結 |

## 処理フロー

```
ユーザークエリ（1件）
      │
      ▼
  Ollama（アプリ層）
  クエリを複数の Cypher に分解
      │
   ┌──┴──┬──────┐
   ▼     ▼      ▼
 クエリ① ② ③  ← Neo4j（DB層）各クエリを個別に実行
   └──┬──┴──────┘
      ▼
  Ollama（アプリ層）
  結果を統合して最終回答
```

⚠️ **注意**: 各 STEP を上から順番に実行してください。Neo4j 設定セル（STEP 1-2）は1セッション内で2回実行しないでください。

---
## STEP 1 — Neo4j インストール
> ⏱️ 約2〜3分

In [ ]:
%%bash
apt-get install -y openjdk-17-jdk-headless zstd -qq
wget -q https://dist.neo4j.org/neo4j-community-5.18.0-unix.tar.gz
tar -xzf neo4j-community-5.18.0-unix.tar.gz
mv neo4j-community-5.18.0 /opt/neo4j
echo "✅ Neo4j インストール完了"

---
## STEP 2 — Neo4j 設定と起動

⚠️ このセルは **1セッションにつき1回だけ** 実行してください（設定キーが重複するとエラーになります）

In [ ]:
%%bash
conf=/opt/neo4j/conf/neo4j.conf

# grep -q で既存キーを確認してから追記（重複防止）
grep -q 'dbms.security.auth_enabled'      $conf || echo 'dbms.security.auth_enabled=false'             >> $conf
grep -q 'server.bolt.listen_address'      $conf || echo 'server.bolt.listen_address=0.0.0.0:7687'      >> $conf
grep -q 'server.http.listen_address'      $conf || echo 'server.http.listen_address=0.0.0.0:7474'      >> $conf
grep -q 'server.default_listen_address'   $conf || echo 'server.default_listen_address=0.0.0.0'        >> $conf

# 設定検証
/opt/neo4j/bin/neo4j-admin server validate-config
echo "✅ 設定検証完了"

In [ ]:
import subprocess, time, os, socket

env = os.environ.copy()
env['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'

subprocess.Popen(
    ['/opt/neo4j/bin/neo4j', 'console'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env=env
)
print('Neo4j 起動中...')

# Bolt ポートが開くまでポーリング（最大90秒）
for i in range(9):
    time.sleep(10)
    try:
        s = socket.create_connection(('localhost', 7687), timeout=3)
        s.close()
        print(f'✅ Neo4j 起動完了（{(i+1)*10}秒）')
        break
    except:
        print(f'  待機中... {(i+1)*10}秒経過')
else:
    print('❌ 起動タイムアウト。ログを確認してください')
    r = subprocess.run(['tail', '-20', '/opt/neo4j/logs/neo4j.log'], capture_output=True, text=True)
    print(r.stdout)

---
## STEP 3 — Ollama インストールとモデルダウンロード
> ⏱️ llama3 は約 4.7GB。`phi3`（軽量版・約2.3GB）に変更も可能

In [ ]:
%%bash
curl -fsSL https://ollama.com/install.sh | sh
echo "✅ Ollama インストール完了"
ollama --version

In [ ]:
import subprocess, time, os

# Ollama サーバー起動
env = os.environ.copy()
env['OLLAMA_HOST'] = '0.0.0.0'
subprocess.Popen(['ollama', 'serve'], env=env,
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print('Ollama サーバー起動完了')

# ── モデル選択 ──────────────────────────────────────
MODEL_NAME = 'llama3'   # 軽量版にする場合は 'phi3' に変更
# ────────────────────────────────────────────────────

print(f"モデル '{MODEL_NAME}' をダウンロード中（数分かかります）...")
subprocess.run(['ollama', 'pull', MODEL_NAME])
print(f"✅ モデル '{MODEL_NAME}' 準備完了")

---
## STEP 4 — ライブラリインストールと日本語フォント設定

In [ ]:
!pip install neo4j requests networkx matplotlib -q
print('✅ ライブラリインストール完了')

In [ ]:
import os, subprocess

subprocess.run(['apt-get', 'install', '-y', 'fonts-ipafont'], capture_output=True)

# matplotlibキャッシュを書き込み可能な場所に設定
os.environ['MPLCONFIGDIR'] = '/tmp/matplotlib'
os.makedirs('/tmp/matplotlib', exist_ok=True)

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

FONT_PATH = '/usr/share/fonts/opentype/ipafont-gothic/ipagp.ttf'
fm.fontManager.addfont(FONT_PATH)
plt.rcParams['font.family'] = 'IPAPGothic'

# テスト描画
fig, ax = plt.subplots(figsize=(4, 1.5))
ax.text(0.5, 0.5, 'グラフRAGデモ：日本語表示OK', ha='center', va='center', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()
print('✅ 日本語フォント設定完了（IPAPGothic）')

---
## STEP 5 — Neo4j 接続とサンプルデータ投入

以下のような人物・組織・プロジェクトの関係グラフを構築します。

```
（田中太郎）─[BELONGS_TO]─▶（開発部）─[PART_OF]─▶（株式会社ABC）
（鈴木一郎）─[BELONGS_TO]─▶（開発部）               ▲
（高橋美咲）─[BELONGS_TO]─▶（開発部）   [REPRESENTS]│
（佐藤花子）─[BELONGS_TO]─▶（営業部）─[PART_OF]─▶（株式会社ABC）
                               ▲ ▲
              [MANAGES]────────┘ └────[MANAGES]
                    （山田次郎）─[REPRESENTS]─▶（株式会社ABC）
```

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver('bolt://localhost:7687', auth=None)

def run_cypher(query: str, params: dict = {}):
    with driver.session() as session:
        return [dict(r) for r in session.run(query, params)]

# 接続テスト
run_cypher('RETURN 1 AS test')
print('✅ Neo4j 接続成功')

In [ ]:
# 既存データ削除
run_cypher('MATCH (n) DETACH DELETE n')

setup_queries = [
    # 人物ノード
    "CREATE (:Person {name: '田中太郎', role: 'engineer'})",
    "CREATE (:Person {name: '佐藤花子', role: 'manager'})",
    "CREATE (:Person {name: '鈴木一郎', role: 'engineer'})",
    "CREATE (:Person {name: '山田次郎', role: 'director'})",
    "CREATE (:Person {name: '高橋美咲', role: 'engineer'})",
    # 組織ノード
    "CREATE (:Org {name: '開発部'})",
    "CREATE (:Org {name: '営業部'})",
    # 会社ノード
    "CREATE (:Company {name: '株式会社ABC'})",
    # プロジェクトノード
    "CREATE (:Project {name: 'プロジェクトX', status: 'active'})",
    "CREATE (:Project {name: 'プロジェクトY', status: 'completed'})",
    # 所属関係
    "MATCH (p:Person {name:'田中太郎'}),(o:Org {name:'開発部'})    CREATE (p)-[:BELONGS_TO]->(o)",
    "MATCH (p:Person {name:'鈴木一郎'}),(o:Org {name:'開発部'})    CREATE (p)-[:BELONGS_TO]->(o)",
    "MATCH (p:Person {name:'高橋美咲'}),(o:Org {name:'開発部'})    CREATE (p)-[:BELONGS_TO]->(o)",
    "MATCH (p:Person {name:'佐藤花子'}),(o:Org {name:'営業部'})    CREATE (p)-[:BELONGS_TO]->(o)",
    # 管理関係
    "MATCH (p:Person {name:'山田次郎'}),(o:Org {name:'開発部'})    CREATE (p)-[:MANAGES]->(o)",
    "MATCH (p:Person {name:'山田次郎'}),(o:Org {name:'営業部'})    CREATE (p)-[:MANAGES]->(o)",
    # 会社所属
    "MATCH (o:Org {name:'開発部'}),(c:Company {name:'株式会社ABC'})  CREATE (o)-[:PART_OF]->(c)",
    "MATCH (o:Org {name:'営業部'}),(c:Company {name:'株式会社ABC'})  CREATE (o)-[:PART_OF]->(c)",
    # 代表関係
    "MATCH (p:Person {name:'山田次郎'}),(c:Company {name:'株式会社ABC'}) CREATE (p)-[:REPRESENTS]->(c)",
    # プロジェクト参加
    "MATCH (p:Person {name:'田中太郎'}),(j:Project {name:'プロジェクトX'}) CREATE (p)-[:WORKS_ON]->(j)",
    "MATCH (p:Person {name:'鈴木一郎'}),(j:Project {name:'プロジェクトX'}) CREATE (p)-[:WORKS_ON]->(j)",
    "MATCH (p:Person {name:'高橋美咲'}),(j:Project {name:'プロジェクトY'}) CREATE (p)-[:WORKS_ON]->(j)",
    "MATCH (p:Person {name:'佐藤花子'}),(j:Project {name:'プロジェクトY'}) CREATE (p)-[:WORKS_ON]->(j)",
]
for q in setup_queries:
    run_cypher(q)

nodes = run_cypher('MATCH (n) RETURN count(n) AS total')[0]['total']
edges = run_cypher('MATCH ()-[r]->() RETURN count(r) AS total')[0]['total']
print(f'✅ データ投入完了 — ノード: {nodes}件 / リレーション: {edges}件')

---
## STEP 6 — グラフの可視化

In [ ]:
import os; os.environ['MPLCONFIGDIR'] = '/tmp/matplotlib'
import matplotlib.font_manager as fm, matplotlib.pyplot as plt
fm.fontManager.addfont('/usr/share/fonts/opentype/ipafont-gothic/ipagp.ttf')
plt.rcParams['font.family'] = 'IPAPGothic'

import networkx as nx
from matplotlib.patches import Patch

G_vis = nx.DiGraph()
for row in run_cypher('MATCH (n) RETURN labels(n)[0] AS label, n.name AS name'):
    G_vis.add_node(row['name'], label=row['label'])
for row in run_cypher('MATCH (a)-[r]->(b) RETURN a.name AS src, type(r) AS rel, b.name AS dst'):
    G_vis.add_edge(row['src'], row['dst'], rel=row['rel'])

color_map = {'Person': '#7F77DD', 'Org': '#1D9E75', 'Company': '#EF9F27', 'Project': '#D85A30'}
node_colors = [color_map.get(G_vis.nodes[n].get('label', ''), '#aaa') for n in G_vis.nodes]

pos = nx.spring_layout(G_vis, seed=42, k=3.0)
plt.figure(figsize=(13, 8))
nx.draw_networkx_nodes(G_vis, pos, node_color=node_colors, node_size=2000, alpha=0.9)
nx.draw_networkx_labels(G_vis, pos, font_size=9, font_color='white',
                        font_weight='bold', font_family='IPAPGothic')
nx.draw_networkx_edges(G_vis, pos, arrows=True, arrowsize=20,
                       edge_color='#888', connectionstyle='arc3,rad=0.1', width=1.5)
nx.draw_networkx_edge_labels(G_vis, pos, nx.get_edge_attributes(G_vis, 'rel'),
                              font_size=7, font_color='#444', font_family='IPAPGothic')
plt.legend(handles=[Patch(color=v, label=k) for k, v in color_map.items()], loc='upper left')
plt.title('Neo4j グラフDB の構造', fontsize=13)
plt.axis('off')
plt.tight_layout()
plt.show()

---
## STEP 7 — Graph RAG パイプラインの定義

**ここがデモの核心部分です。**

- `plan_cypher_queries()` : Ollama（アプリ層）がユーザークエリを複数の Cypher に分解
- `run_graph_rag()` : 分解されたクエリを Neo4j に1件ずつ投入し、結果を統合して回答生成
- Neo4j は受け取ったクエリを実行するだけで、クエリ分解には関与しない

In [ ]:
import requests, json, re

OLLAMA_URL = 'http://localhost:11434/api/generate'

# プロンプト：{{ }} で波括弧をエスケープ（.format() の誤認識を防ぐ）
PLANNER_PROMPT = """あなたはNeo4jグラフデータベースのクエリプランナーです。
ユーザーの質問を、必ず複数の単純なCypherクエリに分解してください。
1つのクエリで複数のホップをたどることは禁止です。必ず1クエリ1ステップに分けてください。

グラフスキーマ:
- ノード: (:Person {{name, role}}), (:Org {{name}}), (:Company {{name}}), (:Project {{name, status}})
- リレーション: BELONGS_TO, MANAGES, PART_OF, REPRESENTS, WORKS_ON

ルール:
- 1つのMATCH文でたどれるリレーションは1種類だけ
- 前のクエリの結果を使う場合はWHERE n.name IN [...]で絞り込む
- 必ず2つ以上のクエリに分解すること

例：「田中太郎が所属する組織の代表は？」の場合:
[
  {{"cypher": "MATCH (p:Person {{name:'田中太郎'}})-[:BELONGS_TO]->(o:Org) RETURN o.name AS org", "purpose": "田中太郎の所属組織を取得"}},
  {{"cypher": "MATCH (p:Person)-[:MANAGES]->(o:Org) WHERE o.name IN ['開発部'] RETURN p.name AS manager", "purpose": "組織のマネージャーを取得"}},
  {{"cypher": "MATCH (p:Person)-[:REPRESENTS]->(c:Company) RETURN p.name AS representative, c.name AS company", "purpose": "会社の代表者を取得"}}
]

JSONの配列のみ返してください。説明不要。

質問: {question}"""


def call_ollama(prompt: str) -> str:
    resp = requests.post(OLLAMA_URL, json={
        'model': MODEL_NAME,
        'prompt': prompt,
        'stream': False,
        'options': {'temperature': 0.0}
    }, timeout=180)
    resp.raise_for_status()
    return resp.json()['response']


def plan_cypher_queries(user_question: str) -> list:
    """LLM（アプリ層）がユーザークエリを複数の Cypher クエリに分解する"""
    raw = call_ollama(PLANNER_PROMPT.format(question=user_question)).strip()
    raw = re.sub(r'```[a-z]*', '', raw).replace('```', '').strip()
    # 配列形式 [...] と辞書形式 {"subqueries":[...]} の両方に対応
    array_match = re.search(r'\[.*\]', raw, re.DOTALL)
    dict_match  = re.search(r'\{.*\}', raw, re.DOTALL)
    if array_match:
        return json.loads(array_match.group())
    elif dict_match:
        return json.loads(dict_match.group()).get('subqueries', [])
    else:
        raise ValueError(f'JSONが見つかりません:\n{raw}')


def run_graph_rag(user_question: str) -> dict:
    """
    Graph RAG パイプライン
      [アプリ層] Ollama がクエリを複数の Cypher に分解
      [DB 層]   Neo4j に1件ずつ投入・実行
      [アプリ層] Ollama が結果を統合して最終回答を生成
    """
    sep = '=' * 65
    print(f'\n{sep}')
    print(f' ユーザークエリ: {user_question}')
    print(sep)

    # ── Step 1: LLM がクエリを計画（アプリ層）────────────────────────
    print('\n[STEP A] Ollama（アプリ層）がクエリを分解中...')
    subqueries = plan_cypher_queries(user_question)
    print(f'  → {len(subqueries)} 件の Cypher クエリを生成')
    for i, sq in enumerate(subqueries, 1):
        print(f'\n  ┌ クエリ {i}/{len(subqueries)}')
        print(f'  │ 目的   : {sq.get("purpose", "")}')
        print(f'  │ Cypher : {sq.get("cypher", "")}')
        print(f'  └' + '─' * 50)

    # ── Step 2: Neo4j に各クエリを個別投入（ここが「複数クエリ」のポイント）
    print('\n[STEP B] Neo4j（DB層）に各 Cypher クエリを1件ずつ投入...')
    print('  ↑ クエリ分解はアプリ層（Ollama）が実施。Neo4j は1件ずつ受け取るだけ')

    all_results = []
    accumulated = {}  # 前クエリの結果を蓄積（プレースホルダ置換用）

    for i, sq in enumerate(subqueries, 1):
        cypher = sq.get('cypher', '')

        # 前クエリの結果で {xxx} プレースホルダを置換
        for key, values in accumulated.items():
            placeholder = '{' + key + '}'
            if placeholder in cypher:
                values_str = ', '.join([f"'{v}'" for v in values])
                cypher = cypher.replace(f"['{placeholder}']", f'[{values_str}]')
                cypher = cypher.replace(placeholder, values_str)

        print(f'\n  Neo4j クエリ {i}/{len(subqueries)}: {cypher}')
        try:
            result = run_cypher(cypher)
        except Exception as e:
            result = [{'error': str(e)}]
        print(f'  結果: {result}')

        # 結果を次クエリ用に蓄積
        for row in result:
            for col, val in row.items():
                accumulated.setdefault(col, [])
                if str(val) not in accumulated[col]:
                    accumulated[col].append(str(val))

        all_results.append({'purpose': sq.get('purpose'), 'cypher': cypher, 'result': result})

    # ── Step 3: LLM が結果を統合して回答生成（アプリ層）──────────────
    print('\n[STEP C] Ollama（アプリ層）が結果を統合して回答を生成中...')
    context = json.dumps(all_results, ensure_ascii=False, indent=2)
    answer = call_ollama(
        f'以下のNeo4jクエリ結果をもとに、質問に日本語で簡潔に答えてください。\n'
        f'質問: {user_question}\n\nクエリ結果:\n{context}\n\n回答:'
    ).strip()

    print(f'\n{sep}')
    print(f' 最終回答: {answer}')
    print(sep)
    print(f'\n 📊 Neo4j へのクエリ数: {len(subqueries)} 件')
    print(f'    （1つのユーザークエリが {len(subqueries)} つの Cypher クエリに展開されました）')
    return {'answer': answer, 'query_count': len(subqueries)}


# 接続テスト
print('Ollama 接続テスト:', call_ollama("'OK'とだけ答えてください")[:20])
print('✅ パイプライン定義完了')

---
## STEP 8 — デモ実行 ① : 2ホップの質問

**「田中太郎が所属する組織を管理している人は誰ですか？」**

期待されるクエリ分解:
1. 田中太郎の所属組織を取得（`BELONGS_TO`）
2. その組織を管理している人を取得（`MANAGES`）

In [ ]:
result1 = run_graph_rag('田中太郎が所属する組織を管理している人は誰ですか？')

---
## STEP 9 — デモ実行 ② : 3ホップの質問

**「田中太郎が所属する組織が属する会社の代表者は誰ですか？」**

期待されるクエリ分解:
1. 田中太郎の所属組織を取得（`BELONGS_TO`）
2. その組織が属する会社を取得（`PART_OF`）
3. その会社の代表者を取得（`REPRESENTS`）

In [ ]:
result2 = run_graph_rag('田中太郎が所属する組織が属する会社の代表者は誰ですか？')

---
## STEP 10 — デモ実行 ③ : 広範囲の質問

**「開発部のメンバー全員と、各メンバーが参加しているプロジェクトを教えてください」**

期待されるクエリ分解:
1. 開発部のメンバー一覧を取得（`BELONGS_TO`）
2. 各メンバーのプロジェクトを取得（`WORKS_ON`）

In [ ]:
result3 = run_graph_rag('開発部のメンバー全員と、各メンバーが参加しているプロジェクトを教えてください')

---
## STEP 11 — 結果の比較グラフ

質問の複雑さ（ホップ数）に応じて Neo4j へのクエリ数が増加することを確認します。

In [ ]:
import os; os.environ['MPLCONFIGDIR'] = '/tmp/matplotlib'
import matplotlib.font_manager as fm, matplotlib.pyplot as plt
fm.fontManager.addfont('/usr/share/fonts/opentype/ipafont-gothic/ipagp.ttf')
plt.rcParams['font.family'] = 'IPAPGothic'

questions = ['Q1\n2ホップ\n（組織の管理者は？）',
             'Q2\n3ホップ\n（会社の代表者は？）',
             'Q3\n広範囲\n（メンバーとPJは？）']
counts = [result1['query_count'], result2['query_count'], result3['query_count']]

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.bar(questions, counts, color=['#7F77DD', '#1D9E75', '#D85A30'], width=0.5)
ax.bar_label(bars, labels=[f'{v}件' for v in counts], padding=5, fontsize=14)
ax.set_ylabel('Neo4j への Cypher クエリ数', fontsize=12)
ax.set_title('1つのユーザークエリ → Neo4j に投げられる Cypher クエリ数\n'
             '（クエリ分解はアプリ層の Ollama が実施）', fontsize=12)
ax.set_ylim(0, max(counts) + 2)
ax.axhline(y=1, color='gray', linestyle='--', alpha=0.5, label='クエリ数=1（単純なDB検索）')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('graph_rag_result.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n【まとめ】')
print('・Ollama（ローカルLLM）がユーザークエリを Cypher クエリに分解  ← アプリ層')
print('・Neo4j は受け取った Cypher を実行するだけ                    ← DB 層')
print('・質問が複雑（ホップ数が多い）ほどクエリ数が増加')

---
## STEP 12 — Neo4j に直接 Cypher を投げて確認（おまけ）

LLM を介さず、Cypher クエリを直接 Neo4j に投げて結果を確認できます。

In [ ]:
print('=== 全 Person ノード ===')
print(run_cypher('MATCH (p:Person) RETURN p.name AS name, p.role AS role'))

print('\n=== 田中太郎の所属組織（1ホップ）===')
print(run_cypher(
    "MATCH (p:Person {name:'田中太郎'})-[:BELONGS_TO]->(o:Org) RETURN o.name AS org"
))

print('\n=== 開発部メンバーの参加プロジェクト（2ホップ）===')
print(run_cypher("""
    MATCH (p:Person)-[:BELONGS_TO]->(:Org {name:'開発部'})
    MATCH (p)-[:WORKS_ON]->(j:Project)
    RETURN p.name AS person, j.name AS project, j.status AS status
"""))

print('\n=== 田中太郎 → 組織 → 会社 → 代表（3ホップを1クエリで）===')
print(run_cypher("""
    MATCH (p:Person {name:'田中太郎'})-[:BELONGS_TO]->(o:Org)
          -[:PART_OF]->(c:Company)<-[:REPRESENTS]-(rep:Person)
    RETURN p.name AS person, o.name AS org,
           c.name AS company, rep.name AS representative
"""))
print('\n↑ Graph RAG では上記を LLM が3つのクエリに分解して順番に実行します')